Como ficará o novo código?
Teremos quatro células principais:
- 1. Configuração: tabelas, chaves e conversões necessárias.
- 2. Transformação: leitura e tratamento das oito tabelas.
- 3. Validação: verificações essenciais.
- 4. Publicação: gravação controlada em Delta Lake.

In [0]:
# O QUE: Configurar as oito tabelas da Silver.
# POR QUÊ: Evitar repetir nomes e regras em vários códigos.

tabelas = {
    "customers": ["customer_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "geolocation": [],
    "product_category_name_translation": ["product_category_name"]
}

print("Tabelas configuradas:", len(tabelas))

for nome, chave in tabelas.items():
    print(nome, "→", chave)

In [0]:
# O QUE: Definir as conversões necessárias.
# POR QUÊ: Evitar repetir os tratamentos para cada tabela.

from pyspark.sql import functions as F

tipos = {
    "order_items": {
        "order_item_id": "int",
        "price": "decimal(12,2)",
        "freight_value": "decimal(12,2)",
        "shipping_limit_date": "timestamp"
    },
    "order_payments": {
        "payment_sequential": "int",
        "payment_installments": "int",
        "payment_value": "decimal(12,2)"
    },
    "order_reviews": {
        "review_score": "int",
        "review_creation_date": "timestamp",
        "review_answer_timestamp": "timestamp"
    },
    "products": {
        "product_name_lenght": "int",
        "product_description_lenght": "int",
        "product_photos_qty": "int",
        "product_weight_g": "int",
        "product_length_cm": "int",
        "product_height_cm": "int",
        "product_width_cm": "int"
    },
    "geolocation": {
        "geolocation_lat": "double",
        "geolocation_lng": "double"
    }
}

print("Regras de conversão configuradas:", len(tipos))

In [0]:
# O QUE: Aplicar os tratamentos nas oito tabelas.
# POR QUÊ: Evitar repetir o mesmo código para cada tabela.

from pyspark.sql import functions as F

dados_silver = {}

for nome in tabelas:

    # Ler a tabela Bronze
    df = spark.table(f"workspace.bronze.{nome}")

    # Aplicar as conversões configuradas
    for coluna, tipo in tipos.get(nome, {}).items():
        df = df.withColumn(coluna, F.col(coluna).cast(tipo))

    # Padronizar cidades e estados
    if nome == "customers":
        df = df.withColumn("customer_city", F.lower(F.trim("customer_city")))
        df = df.withColumn("customer_state", F.upper(F.trim("customer_state")))

    if nome == "sellers":
        df = df.withColumn("seller_city", F.lower(F.trim("seller_city")))
        df = df.withColumn("seller_state", F.upper(F.trim("seller_state")))

    # Guardar o resultado para validação posterior
    dados_silver[nome] = df

    print(f"{nome}: transformação preparada")

print("Total de tabelas preparadas:", len(dados_silver))

In [0]:
# O QUE: Validar as oito tabelas preparadas.
# POR QUÊ: Detectar perda de registros e campos obrigatórios nulos.

from pyspark.sql import functions as F

quantidades = {
    "customers": 99441,
    "order_items": 112650,
    "order_payments": 103886,
    "order_reviews": 99224,
    "products": 32951,
    "sellers": 3095,
    "geolocation": 1000163,
    "product_category_name_translation": 71
}

campos_obrigatorios = {
    "customers": ["customer_id"],
    "order_items": ["order_id", "order_item_id", "price", "freight_value", "shipping_limit_date"],
    "order_payments": ["order_id", "payment_sequential", "payment_value"],
    "order_reviews": ["review_score", "review_creation_date", "review_answer_timestamp"],
    "products": ["product_id"],
    "sellers": ["seller_id"]
}

for nome, df in dados_silver.items():

    campos = campos_obrigatorios.get(nome, [])

    verificacoes = [F.count("*").alias("total")]

    for campo in campos:
        verificacoes.append(
            F.count(F.when(F.col(campo).isNull(), 1)).alias(campo)
        )

    resultado = df.agg(*verificacoes).first()

    assert resultado["total"] == quantidades[nome], f"Quantidade incorreta: {nome}"

    for campo in campos:
        assert resultado[campo] == 0, f"Nulos em {nome}.{campo}"

    print(f"{nome}: OK ({resultado['total']} registros)")

print("Validações essenciais concluídas!")

In [0]:
# O QUE: Publicar as tabelas Silver em Delta Lake.
# POR QUÊ: Criar apenas as tabelas que ainda não existem.

for nome, df in dados_silver.items():

    tabela = f"workspace.silver.{nome}"

    if spark.catalog.tableExists(tabela):
        print(f"{nome}: já existe, preservada")
        continue

    df.write.format("delta").saveAsTable(tabela)

    print(f"{nome}: criada com sucesso")

print("Publicação Silver finalizada!")